<a href="https://colab.research.google.com/github/matcha-jillyace/FORMELOZA_3B_CSST101/blob/main/CSST101_Set3_Formeloza_JillianRose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**#SET 3: Dengue Prevention Advisory KRR System**

# PART 1: KNOWLEDGE BASE (IF-THEN RULES)  -- 9 rules
# Each rule: id, "if" (all conditions must be known), "then" (conclusion)
**

In [43]:
rules <- list(
  list(id = "R1", `if` = c("standing_water"),
       then = "potential_breeding_site"),
  list(id = "R2", `if` = c("potential_breeding_site", "uncovered_containers"),
       then = "recommend_source_reduction"),
  list(id = "R3", `if` = c("multiple_breeding_sites"),
       then = "environmental_concern"),
  list(id = "R4", `if` = c("environmental_concern"),
       then = "recommend_community_inspection"),
  list(id = "R5", `if` = c("recommend_community_inspection"),
       then = "generate_inspection_advisory"),
  list(id = "R6", `if` = c("clogged_drains"),
       then = "potential_breeding_site"),
  list(id = "R7", `if` = c("uncollected_garbage", "recent_rain"),
       then = "potential_breeding_site"),
  list(id = "R8", `if` = c("potential_breeding_site", "recent_rain"),
       then = "environmental_concern"),
  list(id = "R9", `if` = c("recommend_source_reduction"),
       then = "generate_cleanup_advisory")
)

fact_labels <- c(
  standing_water                 = "Standing water observed",
  uncovered_containers           = "Uncovered containers observed",
  multiple_breeding_sites        = "Multiple breeding sites reported",
  clogged_drains                 = "Clogged drains observed",
  uncollected_garbage            = "Uncollected garbage observed",
  recent_rain                    = "Recent rainfall reported",
  potential_breeding_site        = "Potential breeding site",
  recommend_source_reduction     = "Recommend source reduction",
  environmental_concern          = "Environmental concern",
  recommend_community_inspection = "Recommend community inspection",
  generate_inspection_advisory   = "Generate inspection advisory",
  generate_cleanup_advisory      = "Generate clean-up advisory"
)
label <- function(x) ifelse(x %in% names(fact_labels), fact_labels[x], x)

PART 2: INFERENCE ENGINE (FORWARD CHAINING)
# Keeps firing rules until no NEW facts are inferred.


In [44]:
forward_chain <- function(observed, rules) {
  facts <- unique(observed)
  trace <- data.frame(step = integer(), rule = character(),
                      conditions = character(), conclusion = character(),
                      stringsAsFactors = FALSE)
  step <- 0
  repeat {
    new_fact_added <- FALSE
    for (r in rules) {
      all_true <- all(r$`if` %in% facts)
      is_new   <- !(r$then %in% facts)
      if (all_true && is_new) {
        facts <- c(facts, r$then)
        step <- step + 1
        trace[nrow(trace) + 1, ] <- list(step, r$id,
                                         paste(r$`if`, collapse = " AND "),
                                         r$then)
        new_fact_added <- TRUE
      }
    }
    if (!new_fact_added) break
  }
  list(observed = unique(observed),
       inferred = setdiff(facts, unique(observed)),
       trace = trace)
}


PART 3: EXPLANATION + ADVISORY OUTPUT


In [45]:
run_scenario <- function(name, observed) {

  cat("SCENARIO:", name, "\n")

  res <- forward_chain(observed, rules)

  cat("\n[OBSERVED FACTS]\n")
  if (length(res$observed) == 0) cat("  (none reported)\n")
  for (f in res$observed) cat("  -", label(f), "\n")

  cat("\n[INFERRED CONCLUSIONS]\n")
  if (length(res$inferred) == 0) cat("  (none inferred)\n")
  for (f in res$inferred) cat("  *", label(f), "\n")

  cat("\n[REASONING TRACE]\n")
  if (nrow(res$trace) == 0) {
    cat("  No rule fired.\n")
  } else {
    for (i in seq_len(nrow(res$trace))) {
      tr <- res$trace[i, ]
      cat(sprintf("  Step %d | %s: IF %s THEN %s\n", tr$step, tr$rule,
                  paste(label(strsplit(tr$conditions, " AND ")[[1]]), collapse = " AND "),
                  label(tr$conclusion)))
    }
  }

  cat("\n[WHICH RULE PRODUCED EACH CONCLUSION]\n")
  if (nrow(res$trace) == 0) {
    cat("  (none)\n")
  } else {
    for (i in seq_len(nrow(res$trace)))
      cat(sprintf("  %s  <--  %s\n", label(res$trace$conclusion[i]), res$trace$rule[i]))
  }

  cat("\n[ADVISORY]\n")
  if ("generate_inspection_advisory" %in% res$inferred)
    cat("  Inspection advisory: Community inspection is recommended due to environmental concern.\n")
  if ("generate_cleanup_advisory" %in% res$inferred)
    cat("  Clean-up advisory: Remove or cover containers and drain standing water (source reduction).\n")

  key_facts <- c("standing_water", "clogged_drains",
                 "uncollected_garbage", "multiple_breeding_sites")

  if (!any(c("generate_inspection_advisory", "generate_cleanup_advisory") %in% res$inferred)) {
    if (length(res$observed) == 0) {
      cat("  Incomplete information: no observations given. Please collect more data.\n")
    } else if (!any(key_facts %in% res$observed)) {
      cat("  No standing water or other breeding-source observation reported.\n")
      cat("  No breeding site can be inferred; information may be incomplete.\n")
    } else {
      cat("  No advisory generated from the current observations.\n")
    }
  }
  cat("  (Illustrative only. Not a diagnosis or validated risk estimate.)\n")
  invisible(res)
}

# PART 4: TEST SCENARIOS (6 scenarios)


In [46]:
run_scenario("1. Standing water + uncovered containers",
             c("standing_water", "uncovered_containers"))

run_scenario("2. Multiple breeding sites reported",
             c("multiple_breeding_sites"))

run_scenario("3. No standing water (only uncovered containers)",
             c("uncovered_containers"))

run_scenario("4. Incomplete information (no observations)",
             character(0))

run_scenario("5. Clogged drains + recent rain",
             c("clogged_drains", "recent_rain"))

run_scenario("6. Duplicate facts + garbage + rain + containers",
             c("standing_water", "standing_water", "uncollected_garbage",
               "recent_rain", "uncovered_containers"))

run_scenario("7. Uncollected garbage + recent rain (tests R7)",
             c("uncollected_garbage", "recent_rain"))

run_scenario("8. Incomplete information (rain only)",
             c("recent_rain"))

SCENARIO: 1. Standing water + uncovered containers 

[OBSERVED FACTS]
  - Standing water observed 
  - Uncovered containers observed 

[INFERRED CONCLUSIONS]
  * Potential breeding site 
  * Recommend source reduction 
  * Generate clean-up advisory 

[REASONING TRACE]
  Step 1 | R1: IF Standing water observed THEN Potential breeding site
  Step 2 | R2: IF Potential breeding site AND Uncovered containers observed THEN Recommend source reduction
  Step 3 | R9: IF Recommend source reduction THEN Generate clean-up advisory

[WHICH RULE PRODUCED EACH CONCLUSION]
  Potential breeding site  <--  R1
  Recommend source reduction  <--  R2
  Generate clean-up advisory  <--  R9

[ADVISORY]
  Clean-up advisory: Remove or cover containers and drain standing water (source reduction).
  (Illustrative only. Not a diagnosis or validated risk estimate.)
SCENARIO: 2. Multiple breeding sites reported 

[OBSERVED FACTS]
  - Multiple breeding sites reported 

[INFERRED CONCLUSIONS]
  * Environmental concern

PART 5: SIMPLE INTERFACE / DEMO
# Use advise(c("fact1", "fact2")) with any facts from the list below.
# Unknown facts are ignored with a warning message.


In [47]:
valid_facts <- c("standing_water", "uncovered_containers",
                 "multiple_breeding_sites", "clogged_drains",
                 "uncollected_garbage", "recent_rain")

advise <- function(observed, title = "Custom input") {
  unknown <- setdiff(observed, valid_facts)
  if (length(unknown) > 0) {
    cat("\nWARNING: unknown fact(s) ignored:", paste(unknown, collapse = ", "), "\n")
    observed <- intersect(observed, valid_facts)
  }
  run_scenario(title, observed)
}


cat("DEMO INTERFACE\n")
cat("Allowed facts:", paste(valid_facts, collapse = ", "), "\n")
cat("Example call: advise(c(\"standing_water\", \"uncovered_containers\"))\n")



advise(c("standing_water", "recent_rain", "banana"), title = "Demo run (with one invalid fact)")



DEMO INTERFACE
Allowed facts: standing_water, uncovered_containers, multiple_breeding_sites, clogged_drains, uncollected_garbage, recent_rain 
Example call: advise(c("standing_water", "uncovered_containers"))

SCENARIO: Demo run (with one invalid fact) 

[OBSERVED FACTS]
  - Standing water observed 
  - Recent rainfall reported 

[INFERRED CONCLUSIONS]
  * Potential breeding site 
  * Environmental concern 
  * Recommend community inspection 
  * Generate inspection advisory 

[REASONING TRACE]
  Step 1 | R1: IF Standing water observed THEN Potential breeding site
  Step 2 | R8: IF Potential breeding site AND Recent rainfall reported THEN Environmental concern
  Step 3 | R4: IF Environmental concern THEN Recommend community inspection
  Step 4 | R5: IF Recommend community inspection THEN Generate inspection advisory

[WHICH RULE PRODUCED EACH CONCLUSION]
  Potential breeding site  <--  R1
  Environmental concern  <--  R8
  Recommend community inspection  <--  R4
  Generate inspection a

# PART 6: LIMITATIONS AND IMPROVEMENT


In [48]:
cat("LIMITATIONS AND IMPROVEMENT\n\n")

cat("Limitation 1: Facts are only TRUE or missing (no 'unknown', 'maybe', or negation),\n")
cat("  so the system cannot tell 'no standing water' from 'not reported'.\n")
cat("  Incomplete information simply produces no conclusion.\n")
cat("Limitation 2: Rules are fixed and hand-written; they have no weights, so\n")
cat("  all conclusions look equally important and nothing is validated.\n")
cat("Improvement : Add confidence/severity scores (e.g., low/medium/high) to\n")
cat("  rules and facts so advisories can be prioritized.\n")

LIMITATIONS AND IMPROVEMENT

Limitation 1: Facts are only TRUE or missing (no 'unknown', 'maybe', or negation),
  so the system cannot tell 'no standing water' from 'not reported'.
  Incomplete information simply produces no conclusion.
Limitation 2: Rules are fixed and hand-written; they have no weights, so
  all conclusions look equally important and nothing is validated.
Improvement : Add confidence/severity scores (e.g., low/medium/high) to
  rules and facts so advisories can be prioritized.
